# Alle Finetune-Schritte vs. Baseline — Gesamtauswertung

Vergleich der bisher trainierten Modelle auf dem Kiel-Testset (kronenweise, IoU 0.5,
**einheitliches Postprocessing 10/1** — apples-to-apples):

| Modell | Trainingsdaten | Kanäle |
|---|---|---|
| `baseline` | un-finetunt (`freudenberg2022`, ~20cm Sommer) | 5 (RGBI+NDVI) |
| `step1_spring75` | 100% Frühjahr **7.5cm** | 5 |
| `step2_spring20` | 100% Frühjahr **20cm** | 5 |

Datenquellen (alle als `nda` erzeugt): `3_Model/runs/baseline_eval.csv`,
`runs/step1_spring75/eval_test.csv`, `runs/step2_spring20/eval_test.csv` (+ die
`train_log.csv` der beiden Finetune-Runs). Erweitert `results_step1_vs_baseline.ipynb`
um Schritt 2.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

R = Path('/home/leafline/leafline/3_Model/runs')
COLORS = {'baseline': '#eda100', 'step1_spring75': '#2a78d6', 'step2_spring20': '#008300'}
RES_ORDER = ['7.5cm', '20cm', '20cm-spring']

def load(p, run=None):
    p = Path(p)
    if not p.exists():
        print(f'FEHLT: {p}'); return None
    df = pd.read_csv(p)
    if run: df['run'] = run
    print(f'geladen: {p.relative_to(R)} ({len(df)} Zeilen)')
    return df

evals = {
    'baseline':       load(R/'baseline_eval.csv', 'baseline'),
    'step1_spring75': load(R/'step1_spring75/eval_test.csv', 'step1_spring75'),
    'step2_spring20': load(R/'step2_spring20/eval_test.csv', 'step2_spring20'),
}
train_logs = {
    'step1_spring75': load(R/'step1_spring75/train_log.csv'),
    'step2_spring20': load(R/'step2_spring20/train_log.csv'),
}

## 1. Trainingsverlauf — pixelweise Val-F1

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
for run, tl in train_logs.items():
    if tl is None: continue
    ax.plot(tl['epoch'], tl['val_f1'], color=COLORS[run], lw=2, label=run)
    bi = tl['val_f1'].idxmax()
    ax.scatter([tl.loc[bi,'epoch']],[tl.loc[bi,'val_f1']], color=COLORS[run], zorder=5)
    print(f"{run}: bestes val_F1 {tl['val_f1'].max():.4f} @ ep{int(tl.loc[bi,'epoch'])}")
ax.set_xlabel('Epoche'); ax.set_ylabel('Val-F1 (pixelweise)')
ax.set_title('Trainingsverlauf'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print('(v1-Referenz mit nDOM: 0.706)')

## 2. Kronenweise Test-F1: Gesamtmatrix (PP 10/1)

Mikro-Schnitt (TP/FP/FN über beide Testgebiete summiert, dann Metrik):

| Auflösung | baseline | step1 (Frühjahr 7.5) | step2 (Frühjahr 20) |
|---|---|---|---|
| **7.5cm** (Frühjahr) | 0.000 | **0.120** | 0.021 |
| **20cm** (Sommer) | **0.340** | 0.052 | 0.008 |
| **20cm-spring** (Frühjahr) | 0.044 | 0.041 | **0.098** |

**Diagonale:** jedes Modell ist auf seiner eigenen Domäne am stärksten (fett).
Live-Neuberechnung aus den CSVs:

In [ ]:
def micro(df):
    out = {}
    for res, g in df.groupby('aufloesung'):
        tp, fp, fn = g.tp.sum(), g.fp.sum(), g.fn.sum()
        p = tp/(tp+fp) if tp+fp else 0.0
        r = tp/(tp+fn) if tp+fn else 0.0
        out[res] = 2*p*r/(p+r) if p+r else 0.0
    return out

tbl = pd.DataFrame({run: micro(df) for run, df in evals.items() if df is not None})
tbl = tbl.reindex(RES_ORDER)
print('Mikro-F1 je Modell × Auflösung:')
print(tbl.round(4).to_string())

## 3. Vergleichsdiagramm — F1 je Auflösung

In [ ]:
runs = [r for r in ['baseline','step1_spring75','step2_spring20'] if r in tbl.columns]
x = np.arange(len(RES_ORDER)); w = 0.26
fig, ax = plt.subplots(figsize=(10,5.5))
for k, run in enumerate(runs):
    vals = [tbl.loc[res, run] if res in tbl.index else 0 for res in RES_ORDER]
    off = (k - (len(runs)-1)/2) * w
    ax.bar(x+off, vals, width=w, label=run, color=COLORS[run])
    for xi, v in zip(x+off, vals):
        ax.text(xi, v+0.005, f'{v:.3f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(RES_ORDER)
ax.set_ylabel('Mikro-F1 (kronenweise, IoU 0.5)')
ax.set_ylim(0, max(0.4, np.nanmax(tbl.values)*1.25))
ax.set_title('Baseline vs. Finetune-Schritte je Auflösung (PP 10/1)')
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

## 4. Befunde & Schedule-Stand

**1. Finetuning ist auf den Frühjahrs-Domänen ein klarer Gewinn.**
- 7.5cm: Baseline 0.000 (praktisch nichts erkannt, OOD) → step1 **0.120**.
- 20cm-spring: Baseline 0.044 / step1 0.041 → step2 **0.098** (> 2×). Genau das im
  Schedule erwartete Ergebnis von Schritt 2.

**2. Jede Domäne braucht ihr eigenes Modell — Diagonale.** step1 ist am besten bei
7.5cm, step2 bei 20cm-spring, die Baseline bei 20cm-Sommer. Kein Modell deckt mehr als
seine Trainingsecke ab.

**3. Die Jahreszeit ist ein eigener, großer Faktor.** Bei *gleicher* 20cm-Auflösung
bricht step2 (Frühjahr) auf Sommer ein (0.008), während es auf Frühjahr 0.098 erreicht;
die Baseline (Sommer) ist umgekehrt bei Sommer stark (0.340), bei Frühjahr schwach (0.044).
Nicht nur Auflösung, auch Saison verursacht je einen großen Domänen-Sprung.

**Einordnung für Kiel** (fliegt Frühjahr): die relevanten Zellen 7.5cm (step1 0.120) und
20cm-spring (step2 0.098) liegen jetzt klar über der Baseline. Die 20cm-Sommer-Spalte ist
ein Robustheitstest — Frühjahrs-Modelle taugen nicht für Sommer.

**Caveat:** absolute Werte weiter niedrig, Postprocessing ungetunt (step2 über-segmentiert
7.5cm stark: 1044 Vorhersagen; 20cm-spring 226 vs. 357 GT ist dagegen vernünftig).
Per-Auflösung getuntes PP würde die On-Domain-Werte anheben.

**Schedule-Stand:** Schritt 1 ✅ (Auflösung zählt), Schritt 2 ✅ (20cm-Frühjahr-Modell hebt
20cm-spring). Offene Optionen: Schritt 3 (50/50 Sommer+Frühjahr 20cm — testet ein
Saison-übergreifendes 20cm-Modell) oder Schritt 4/5 (Höhenkanal nDOM zu den Frühjahrs-
Modellen — Endzweck des Schedules, größter verbleibender Hebel).
